In [1]:
import sys
print(sys.version)

3.11.3 (main, Aug 30 2024, 23:31:08) [GCC 12.3.0]


In [2]:
!which python3

/sw/arch/RHEL8/EB_production/2023/software/Python/3.11.3-GCCcore-12.3.0/bin/python3


In [3]:
!nvidia-smi

Tue Mar  3 11:50:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.48.01              Driver Version: 590.48.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          On  |   00000000:CA:00.0 Off |                    0 |
| N/A   30C    P0             47W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 0. Modules

Load the modules that will be used in this notebook

In [4]:
import os
import json
import torch
import random
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from sklearn.model_selection import train_test_split

In [5]:
sys.path.append("/home/fcool/agent_annotation")

import agent_utils
from agent_utils.utils import train_validate, build_multi_task_splits, rebalance_binary_to_fixed_n

2026-03-03 11:50:31.327996: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-03 11:50:31.348925: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772535031.369104 4088173 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772535031.376466 4088173 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-03 11:50:31.401840: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [6]:
# - test whether utils loaded
agent_utils.test_function()

This is a test. mm_utils package loaded!


In [7]:
CACHE_DIR = '/projects/prjs1308/huggingface/'

In [8]:
TOPIC01_ALLOWED = [
    "NO TOPIC",
    "ECONOMY",
    "CIVIL RIGHTS",
    "HEALTH",
    "AGRICULTURE",
    "LABOR",
    "EDUCATION",
    "ENVIRONMENT",
    "ENERGY",
    "IMMIGRATION",
    "TRANSPORTATION",
    "LAW AND CRIME",
    "SOCIAL WELFARE",
    "HOUSING",
    "DOMESTIC COMMERCE",
    "DEFENSE",
    "TECHNOLOGY",
    "FOREIGN TRADE",
    "INTERNATIONAL AFFAIRS",
    "GOVERNMENT OPERATIONS",
    "PUBLIC LANDS",
    "CULTURE",
    "ETHNICITY"
]

# Mapping from old topic numbers to new topic numbers (as strings)
topic_mapping = {
    0: 'NO TOPIC',
    1: 'ECONOMY',
    2: 'CIVIL RIGHTS',
    3: 'HEALTH',
    4: 'AGRICULTURE',
    5: 'LABOR',
    6: 'EDUCATION',
    7: 'ENVIRONMENT',
    8: 'ENERGY',
    9: 'IMMIGRATION',
    10: 'TRANSPORTATION',
    12: 'LAW AND CRIME',  # Law and Crime
    13: 'SOCIAL WELFARE',  # Social Welfare
    14: 'HOUSING',  # Housing
    15: 'DOMESTIC COMMERCE',  # Domestic Commerce
    16: 'DEFENSE',  # Defense
    17: 'TECHNOLOGY',  # Technology
    18: 'FOREIGN TRADE',  # Foreign Trade
    19: 'INTERNATIONAL AFFAIRS',  # International Affairs
    20: 'GOVERNMENT OPERATIONS',  # Government Operations
    21: 'PUBLIC LANDS',  # Public Lands
    23: 'CULTURE',
    24: 'ETHNICITY'   # Gun control
} # TOPIC MAPPING TO BE APPLIED TO DATAFRAME

# 1. Data

Load the datasets to be used in this implementation.

In [9]:
json_path = Path("/projects/prjs1308/africa_llm_data/africa_jsons/african_videos.json")

with json_path.open("r", encoding="utf-8") as f:
    records = json.load(f)

print("N records:", len(records))
print(records[0])

N records: 1809
{'id': 1712829139429512, 'post_owner.id': 1475949680204358, 'text': 'La jeunesse, c’est l’époque où on n’a pas de temps. Nous devons profiter du temps qui nous est donné et utiliser l’énergie nécessaire pour changer positivement le monde. Les incompétents sont mauvais ! Ils peuvent vous amener à douter de vous. Ils n’aiment pas ceux qui sont compétents. Ils n’aiment pas ceux qui innovent. Ils sont dangereux ! Cherchez la connaissance et travaillez dur pour être compétents ! Prenons conscience de la situation ! Vous verrez par exemple sur les réseaux sociaux, que ceux qui posent des actions concrètes n’ont pas le soutien de leurs frères. Mais quand il s’agit de choses inutiles, vous verrez que la personne aura des millions de vues. Se distraire c’est bien, mais on doit travailler d’abord ! Cherchez la connaissance, sinon, vous passerez du temps à ne rien faire ! Et la jeunesse, c’est la période idéale pour le faire. Lisons ! Interrogeons l’histoire', 'politics': 1.0, 'do

In [10]:
df = pd.json_normalize(records)

# (keep your list/dict -> json-string cleanup if you want)
for col in df.columns:
    if df[col].apply(lambda x: isinstance(x, (list, dict))).any():
        df[col] = df[col].apply(lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, (list, dict)) else x)

# 1) define target cols: everything right of 'text'
text_idx = df.columns.get_loc("text")
target_cols = list(df.columns[text_idx + 1:])

# 2) drop rows where ALL targets are NaN (before split)
mask_all_nan = df[target_cols].isna().all(axis=1)
print("Dropping rows with all targets NaN:", int(mask_all_nan.sum()), "/", len(df))
df = df.loc[~mask_all_nan].reset_index(drop=True)

df["topic01"] = pd.to_numeric(df["topic01"], errors="coerce").astype("Int64").map(topic_mapping)

# 3) split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=1, shuffle=True)

# 4) add targets_json to both (same target_cols list)
def add_targets_json(df_):
    df_["targets_json"] = df_.apply(
        lambda row: json.dumps(
            {c: (None if pd.isna(row[c]) else row[c]) for c in target_cols},
            ensure_ascii=False,
        ),
        axis=1,
    )

add_targets_json(train_df)
add_targets_json(test_df)

print(train_df[["text", "targets_json"]].head(2).to_string(index=False))
print(test_df[["text", "targets_json"]].head(2).to_string(index=False))

Dropping rows with all targets NaN: 617 / 1809
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         

In [11]:
train_df.head()

,id,post_owner.id,text,politics,domestic_politics,foreign_politics,resource_distribution,resource_distribution_by_whom1,resource_distribution_for_whom_ethnic1,resource_distribution_for_whom_region1,...,pro_china,pro_un,pro_mf,pro_democracy,anti_western,national_unity,subgroup_unity,subgroup_unity_text,african_unity,targets_json
644,691888046638760,1140008943927975,وزارة الأسرة تنظّم يوما تثقيفيّا توعويّا حول ا...,1.0,1.0,0.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0,"{""politics"": 1.0, ""domestic_politics"": 1.0, ""f..."
729,798268729590766,1382113209878566,Your word is a lamp to my feet and a light to ...,0.0,NaN,NaN,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0,"{""politics"": 0.0, ""domestic_politics"": null, ""..."
745,499763589271585,2110981272586198,"Statement By His Excellency Dr Nangolo Mbumba,...",1.0,1.0,1.0,1.0,1,NaN,NaN,...,NaN,3.0,NaN,3.0,0.0,1.0,0.0,NaN,1.0,"{""politics"": 1.0, ""domestic_politics"": 1.0, ""f..."
134,1476647823372974,1357378301936474,الذكرى 73 لاستقلال #ليبيا \n#ارفع_العلم\n#أيام...,0.0,NaN,NaN,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,1.0,1.0,Libyan,0.0,"{""politics"": 0.0, ""domestic_politics"": null, ""..."
613,862182456969717,920694622929278,✅استقبل السيد وزير البيئة، حبيب عبيد، الثلاثاء...,1.0,1.0,1.0,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,NaN,0.0,"{""politics"": 1.0, ""domestic_politics"": 1.0, ""f..."


In [12]:
# train_df['topic01'].values

Load the prompts

In [ ]:
# System prompt = codebook (long, static → KV-cached at inference). User prompt = short template from file.
PROMPTS_DIR = Path("/projects/prjs1308/africa_llm_data/prompts")

system_prompt_path = PROMPTS_DIR / "africa_prompt_system.txt"
inference_prompt_path = PROMPTS_DIR / "inference_prompt.txt"
if system_prompt_path.exists():
    system_prompt = system_prompt_path.read_text(encoding="utf-8")
    if inference_prompt_path.exists():
        prompt = inference_prompt_path.read_text(encoding="utf-8").strip()
    else:
        prompt = "Social media post text:\n\n{}\n\nAnnotate this post according to the codebook and return a single JSON object only."
    print("Using system + user prompt split (KV prefix caching enabled).")
    print("System prompt length:", len(system_prompt))
    print("User prompt (from inference_prompt.txt):", prompt[:80], "...")
else:
    system_prompt = None
    # Fallback: single full prompt (no prefix caching)
    prompt_path = PROMPTS_DIR / "africa_prompt_2602.txt"
    prompt = prompt_path.read_text(encoding="utf-8")
    print("Using single prompt (no system_prompt file found).")
    print("Length:", len(prompt))

Text: ‘{}’. This was the text of a social media post. Your ask is to code it on a number of targets:
________________________________________________________________________
1. politics


Binary variable: either 0 or 1, where 1 indicates that the post is about politically relevant content. 


Use the following guidelines when deciding whether the post is about politics:
A post should be considered as politically relevant (so be coded as  1) if:
It’s about any political issue, (potentially) politically relevant topics (including conspiracy theories) are being discussed, such as science, immigration, civil liberties, labor, law & crime, international affairs, education, healthcare, housing, etc.
It’s about a politically relevant (or historical) event/s, political actor/s
It is about develop
Length: 13560


# Define rules and classes

In [14]:
TARGETS = {
    # multiclass
    "language": {"type": "multiclass", "allowed": [1,2,3,4,5,6,7,8,99]},
    "resource_distribution_by_whom1": {"type": "multiclass", "allowed": [3,2,1,0,-1,99]},
    "resource_distribution_for_whom1": {"type": "multiclass", "allowed": [1,0,-1,99]},
    "climate_change": {"type": "multiclass", "allowed": [0,1,2,99]},
    "topic01": {"type": "multiclass", "allowed": TOPIC01_ALLOWED},
    "pro_us": {"type": "multiclass", "allowed": [1,2,3,99]},
    "pro_russia": {"type": "multiclass", "allowed": [1,2,3,99]},
    "pro_china": {"type": "multiclass", "allowed": [1,2,3,99]},
    "pro_un": {"type": "multiclass", "allowed": [1,2,3,99]},
    "pro_imf": {"type": "multiclass", "allowed": [1,2,3,99]},  # (or pro_mf if that's your column)
    "pro_democracy": {"type": "multiclass", "allowed": [1,2,3,99]},

    # binary (optionally allow 99 if your data uses it)
    "politics": {"type": "binary", "allowed": [0,1,99]},
    "domestic_politics": {"type": "binary", "allowed": [0,1,99]},
    "foreign_politics": {"type": "binary", "allowed": [0,1,99]},
    "resource_distribution": {"type": "binary", "allowed": [0,1,99]},
    "resource_distribution_for_gender": {"type": "binary", "allowed": [0,1,99]},
    "anti_western": {"type": "binary", "allowed": [0,1,99]},
    "national_unity": {"type": "binary", "allowed": [0,1,99]},
    "subgroup_unity": {"type": "binary", "allowed": [0,1,99]},
    "african_unity": {"type": "binary", "allowed": [0,1,99]},
    "political_opponents": {"type": "binary", "allowed": [0,1,99]},
    "religion": {"type": "binary", "allowed": [0,1,99]},

    # string fields (allowed filled from train+test+val in next cell)
    "resource_distribution_for_whom_ethnic1": {
        "type": "string",
        "allowed": [],
        "eval": {
            "metric": "exact",              # exact match after normalization
            "normalize": ["strip", "lower", "collapse_ws"],
            "empty_allowed": True,          # because it's only required when a condition holds
            "track_unique_incorrect": True,
            "max_unique_incorrect": 200,
        },
    },
    "resource_distribution_for_whom_region1": {
        "type": "string",
        "allowed": [],
        "eval": {
            "metric": "exact",
            "normalize": ["strip", "lower", "collapse_ws"],
            "empty_allowed": True,
            "track_unique_incorrect": True,
            "max_unique_incorrect": 200,
        },
    },
    "subgroup_unity_text": {
        "type": "string",
        "allowed": [],
        "eval": {
            "metric": "exact",
            "normalize": ["strip", "lower", "collapse_ws"],
            "empty_allowed": True,
            "track_unique_incorrect": True,
            "max_unique_incorrect": 500,
        },
    },
}

In [ ]:
# Set string targets' allowed list from all annotated values in train + test (val is a subset of train)
STRING_TARGETS = [
    "resource_distribution_for_whom_ethnic1",
    "resource_distribution_for_whom_region1",
    "subgroup_unity_text",
]
combined = pd.concat([train_df, test_df], ignore_index=True)
for t in STRING_TARGETS:
    if t not in TARGETS or TARGETS[t].get("type") != "string":
        continue
    if t not in combined.columns:
        continue
    vals = combined[t].dropna().astype(str).str.strip()
    vals = vals[vals != ""]
    unique_vals = sorted(vals.unique().tolist())
    TARGETS[t]["allowed"] = unique_vals
    print(f"{t}: allowed = {len(unique_vals)} values")

## 2. Setup training args

In [ ]:
seeds = [42]
results_dir = '/projects/prjs1308/africa_llm_data/results/testing'
model_dir = '/projects/prjs1308/africa_llm_data/results/testing_models'
batch_size = 1
max_tokens = 4096
early_stopping = 5
epochs = 5
gemma_model = "4b"  # "4b" | "27b" for simple_gemma3

## 2. Experimentation

#### 2.2.2 Gemma3 multimodal finetuned 

In [ ]:
# train_validate(
#     mtype="fine_tuned_gemma3",
#     train_df=train_df,
#     test_df=test_df,
#     text_col="text",
#     target_col="targets_json",
#     prompt=prompt,
#     answer_col="targets_json",
#     train_val_seeds=seeds,
#     val_size=0.2,
#     results_folder=results_dir,
#     model_dir=model_dir,
#     batch_size=batch_size,
#     max_tokens=max_tokens,
#     max_new_tokens=10,
#     cache_dir=CACHE_DIR,
#     local_model=None,          # optional: path to LoRA adapter to resume
#     text_only_res=None,        # optional: path to prior results csv to append to
#     early_stopping_patience=early_stopping,
#     epochs=epochs,
#     learning_rates=[0.0002],
#     targets_spec=TARGETS,
# )

Finetuning Gemma3 (new API):
 - text_col: text
 - answer_col: targets_json
 - multimodal: False
GPU available
Starting to fine-tune Gemma-3


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


[DEBUG] slot_tokens: 103
[DEBUG] bad slot tokens mapped to UNK: 0

SLOT TOKEN SETUP DEBUG
Total slot tokens: 103
Slot tokens sample: ['<@african_unity=0>', '<@african_unity=1>', '<@african_unity=99>', '<@anti_western=0>', '<@anti_western=1>', '<@anti_western=99>', '<@climate_change=0>', '<@climate_change=1>', '<@climate_change=2>', '<@climate_change=99>'] ... ['<@topic01=HOUSING>', '<@topic01=IMMIGRATION>', '<@topic01=INTERNATIONAL AFFAIRS>', '<@topic01=LABOR>', '<@topic01=LAW AND CRIME>', '<@topic01=NO TOPIC>', '<@topic01=PUBLIC LANDS>', '<@topic01=SOCIAL WELFARE>', '<@topic01=TECHNOLOGY>', '<@topic01=TRANSPORTATION>']
Tokenizer unk_token_id: 3
Bad (mapped to UNK) slot tokens: 0

Target: language | type=multiclass | #allowed=9
 allowed[:10]: [1, 2, 3, 4, 5, 6, 7, 8, 99]
 slot_tokens[:5]: ['<@language=1>', '<@language=2>', '<@language=3>', '<@language=4>', '<@language=5>']
 token_ids[:10]: [262161, 262162, 262163, 262164, 262165, 262166, 262167, 262168, 262169]

Target: resource_distri

`torch_dtype` is deprecated! Use `dtype` instead!


Val dataset columns: ['text']
Loading model for LR 0.0002


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Model loaded for fold 0
Sample row keys: dict_keys(['text'])


TypeError: SFTTrainer.__init__() got an unexpected keyword argument 'sft_config'

In [ ]:
train_validate(
    mtype="simple_gemma3",      # <── key change
    train_df=train_df,
    test_df=test_df,
    text_col="text",            # column with transcript/post
    target_col="targets_json",  # not used by simple runner, but fine to keep
    prompt=prompt,              # must contain "{}" once for transcript insertion
    answer_col="targets_json",  # column with JSON answers
    train_val_seeds=seeds,
    val_size=0.2,
    results_folder=results_dir,
    model_dir=model_dir,
    batch_size=batch_size,
    max_tokens=max_tokens,
    max_new_tokens=500,
    cache_dir=CACHE_DIR,
    local_model=None,
    text_only_res=None,
    early_stopping_patience=early_stopping,
    epochs=epochs,
    learning_rates=[0.0001],
    gemma_model=gemma_model,    # "4b" | "27b"
    targets_spec=TARGETS,      # for inference: normalize semicolon-separated multiclass (e.g. topic01)
    system_prompt=system_prompt,  # codebook in system role → KV prefix caching at inference
)